# ⚡ Notebook 4: Optimización de Memoria y Rendimiento en Python para ML
### Curso: Machine Learning Avanzado — Módulo 1

**Objetivo:** Comprender y aplicar técnicas de uso eficiente de memoria para escalar pipelines de ML a datasets de millones de registros.

---
## 🗺️ Contenido
1. Modelo de memoria de Python: reference counting, GC
2. Generators vs Listas: procesamiento lazy de datasets masivos
3. `__slots__`: clases con huella de memoria mínima
4. Tipos de datos NumPy: impacto en memoria de modelos
5. Memory mapping: datasets que no caben en RAM
6. Profiling de memoria: `tracemalloc` y `memory_profiler`
7. Caso aplicado: Pipeline de datos para 10M de registros

---
## 1. Modelo de Memoria de Python

### 📖 Concepto
Python usa **reference counting** + **garbage collector cíclico**:
- Cada objeto tiene un contador de referencias
- Cuando llega a 0, la memoria se libera **inmediatamente**
- El GC maneja ciclos (objetos que se referencian mutuamente)

```
x = np.zeros(1000)   # ref_count(array) = 1
y = x                # ref_count(array) = 2
del x                # ref_count(array) = 1 (NO libera)
del y                # ref_count(array) = 0 → LIBERA
```

**Implicaciones en ML:**
- Arrays temporales en transformaciones acumulan memoria si se referencian
- `del array` + `gc.collect()` en pipelines de alto throughput
- Las **vistas** de NumPy mantienen vivo el array original

In [ ]:
import numpy as np
import sys
import gc
import tracemalloc
from sys import getsizeof

# ── Overhead de objetos Python ────────────────────────────────────────────
print("Tamaño de objetos básicos en Python (overhead):")
print(f"  int        : {sys.getsizeof(42):4d} bytes")
print(f"  float      : {sys.getsizeof(3.14):4d} bytes")
print(f"  str (vacio): {sys.getsizeof(''):4d} bytes")
print(f"  list (vacia): {sys.getsizeof([]):4d} bytes")
print(f"  dict (vacio): {sys.getsizeof({}):4d} bytes")

# Comparación: lista de ints vs numpy array
n = 1_000_000
lista_py = list(range(n))
array_np = np.arange(n, dtype=np.int64)
array_np32 = np.arange(n, dtype=np.int32)

print(f"\nMemoria para {n:,} enteros:")
print(f"  Lista Python    : {sys.getsizeof(lista_py) / 1024**2:.2f} MB + overhead de cada int")
print(f"  NumPy int64     : {array_np.nbytes / 1024**2:.2f} MB")
print(f"  NumPy int32     : {array_np32.nbytes / 1024**2:.2f} MB")
print(f"  Ratio lista/np64: ~{(sys.getsizeof(lista_py) + n*28) / array_np.nbytes:.0f}x más memoria")

# Reference counting
print("\nReference counting:")
A = np.random.randn(1000, 1000)
print(f"  A creado. Ref count: {sys.getrefcount(A) - 1}")  # -1 por arg de getrefcount
B = A  # segunda referencia
print(f"  B = A.  Ref count: {sys.getrefcount(A) - 1}")
C = A[:500, :]  # vista — NO copia, pero mantiene A vivo
print(f"  C = A[:500,:] (vista). Ref count A: {sys.getrefcount(A) - 1}")
del B
print(f"  del B.  Ref count: {sys.getrefcount(A) - 1}")
print(f"  A liberada al hacer del A Y del C (C es vista de A)")

In [ ]:
# ── Generators: Procesamiento Lazy para Datasets Masivos ─────────────────
import time

# PROBLEMA: cargar todo el dataset en memoria
# Simulamos un archivo CSV de 10M de registros

def generate_dataset_eager(n: int):
    """Carga todo en memoria (lista) — problemático para datasets grandes."""
    return [
        {"id": i, "feature1": i * 0.01, "feature2": i % 100, "label": i % 2}
        for i in range(n)
    ]

def generate_dataset_lazy(n: int):
    """Genera registros uno a uno — O(1) memoria en todo momento."""
    for i in range(n):
        yield {"id": i, "feature1": i * 0.01, "feature2": i % 100, "label": i % 2}

def process_in_batches(generator, batch_size: int):
    """Procesa en mini-batches — patrón estándar en Deep Learning."""
    batch = []
    for item in generator:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []  # libera memoria del batch anterior
    if batch:  # último batch parcial
        yield batch

N = 1_000_000
BATCH_SIZE = 10_000

# Comparar memoria: eager vs lazy
tracemalloc.start()
gen = generate_dataset_lazy(N)
current1, peak1 = tracemalloc.get_traced_memory()
tracemalloc.stop()

tracemalloc.start()
data_eager = generate_dataset_eager(10_000)  # solo 10K para no agotar RAM
current2, peak2 = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("Comparación de memoria:")
print(f"  Generator (1M registros): pico = {peak1 / 1024:.1f} KB")
print(f"  Lista (10K registros):    pico = {peak2 / 1024**2:.1f} MB")
print(f"  Relación memoria: x{peak2/peak1:.0f} (con 100x menos datos en lista!)")

# Procesamiento en batches
print(f"\nProcesamiento de {N:,} registros en batches de {BATCH_SIZE:,}:")
t0 = time.time()
total_procesado = 0
suma_features = 0.0

for i, batch in enumerate(process_in_batches(generate_dataset_lazy(N), BATCH_SIZE)):
    # Convertir a NumPy para operaciones vectorizadas
    batch_array = np.array([[r["feature1"], r["feature2"]] for r in batch])
    suma_features += batch_array[:, 0].sum()
    total_procesado += len(batch)

print(f"  Registros procesados: {total_procesado:,}")
print(f"  Tiempo: {time.time() - t0:.2f}s")
print(f"  Suma feature1: {suma_features:.0f}")

In [ ]:
# ── __slots__: Clases con Huella Mínima de Memoria ────────────────────────

# Sin __slots__ (clase normal)
class PuntoDS_Normal:
    def __init__(self, x, y, label, weight=1.0):
        self.x = x
        self.y = y
        self.label = label
        self.weight = weight

# Con __slots__ (huella mínima)
class PuntoDS_Slots:
    __slots__ = ["x", "y", "label", "weight"]

    def __init__(self, x, y, label, weight=1.0):
        self.x = x
        self.y = y
        self.label = label
        self.weight = weight

# Medir memoria de instancias individuales
p_normal = PuntoDS_Normal(1.5, 2.3, 0)
p_slots  = PuntoDS_Slots(1.5, 2.3, 0)

print("Memoria por instancia:")
print(f"  Clase normal: {sys.getsizeof(p_normal)} bytes base + {sys.getsizeof(p_normal.__dict__)} bytes dict")
print(f"  Clase __slots__: {sys.getsizeof(p_slots)} bytes (sin __dict__)")

# Impacto a escala: millones de puntos de datos
tracemalloc.start()
puntos_normal = [PuntoDS_Normal(i*0.01, i*0.02, i%3) for i in range(100_000)]
_, peak_normal = tracemalloc.get_traced_memory()
del puntos_normal
gc.collect()
tracemalloc.stop()

tracemalloc.start()
puntos_slots = [PuntoDS_Slots(i*0.01, i*0.02, i%3) for i in range(100_000)]
_, peak_slots = tracemalloc.get_traced_memory()
del puntos_slots
gc.collect()
tracemalloc.stop()

print(f"\nMemoria para 100K instancias:")
print(f"  Clase normal  : {peak_normal / 1024**2:.2f} MB")
print(f"  Clase __slots__: {peak_slots / 1024**2:.2f} MB")
print(f"  Ahorro: {(1 - peak_slots/peak_normal)*100:.1f}%")

In [ ]:
# ── Downcasting: Reducir Tipos de Datos en Arrays ─────────────────────────
# Técnica estándar en Kaggle y producción para grandes DataFrames

def optimize_numpy_dtypes(array: np.ndarray) -> np.ndarray:
    """Reduce el tipo de dato de un array NumPy al mínimo necesario.
    
    Estrategia:
    - Enteros: usar el tipo más pequeño que contiene el rango
    - Flotantes: usar float32 si la precisión lo permite
    """
    if np.issubdtype(array.dtype, np.integer):
        mn, mx = array.min(), array.max()
        for dtype in [np.int8, np.int16, np.int32, np.int64]:
            info = np.iinfo(dtype)
            if mn >= info.min and mx <= info.max:
                return array.astype(dtype)
    elif np.issubdtype(array.dtype, np.floating):
        # Verificar si float32 es suficiente
        array_f32 = array.astype(np.float32)
        max_error = np.abs(array - array_f32).max()
        if max_error < 1e-4:  # tolerancia
            return array_f32
    return array

# Simulación: dataset de sensores IoT
np.random.seed(42)
dataset = {
    "sensor_id":     np.random.randint(0, 50, 1_000_000),       # int64 → int8
    "temperatura":   np.random.uniform(-20, 120, 1_000_000),    # float64 → float32
    "timestamp":     np.arange(1_000_000, dtype=np.int64),      # se mantiene int64
    "codigo_evento": np.random.randint(0, 32767, 1_000_000),    # int64 → int16
}

mem_original = sum(a.nbytes for a in dataset.values())

dataset_opt = {k: optimize_numpy_dtypes(v) for k, v in dataset.items()}
mem_optimizado = sum(a.nbytes for a in dataset_opt.values())

print("Optimización de tipos de datos — Dataset IoT (1M registros):")
print(f"{'Feature':15} | {'Antes':>8} | {'Después':>10} | {'Memoria':>10}")
print("-" * 55)
for key in dataset:
    before = dataset[key].dtype
    after  = dataset_opt[key].dtype
    mem_b  = dataset[key].nbytes
    mem_a  = dataset_opt[key].nbytes
    print(f"{key:15} | {str(before):>8} | {str(after):>10} | "
          f"{mem_b/1024**2:5.1f}→{mem_a/1024**2:.1f} MB")

print(f"\nTotal:")
print(f"  Antes    : {mem_original / 1024**2:.1f} MB")
print(f"  Después  : {mem_optimizado / 1024**2:.1f} MB")
print(f"  Reducción: {(1 - mem_optimizado/mem_original)*100:.1f}%")

In [ ]:
# ── Memory Mapping: Datasets que no caben en RAM ──────────────────────────
import tempfile
import os

# Simular un dataset grande guardado en disco
n_registros = 5_000_000
n_features  = 20

with tempfile.NamedTemporaryFile(suffix='.npy', delete=False) as f:
    temp_path = f.name

# Crear el archivo en disco (simula datos pre-existentes)
fp = np.memmap(temp_path, dtype=np.float32,
               mode='w+', shape=(n_registros, n_features))
fp[:] = np.random.randn(n_registros, n_features).astype(np.float32)
del fp  # cierra y guarda

# Abrir con memory mapping (NO carga todo en RAM)
datos_mmap = np.memmap(temp_path, dtype=np.float32,
                        mode='r', shape=(n_registros, n_features))

print("Memory-Mapped Array:")
print(f"  Shape: {datos_mmap.shape}")
print(f"  Tamaño en disco: {os.path.getsize(temp_path) / 1024**3:.2f} GB")
print(f"  dtype: {datos_mmap.dtype}")
print(f"  Tipo Python: {type(datos_mmap)}")

# Operar sobre subsets (solo se carga en RAM lo necesario)
tracemalloc.start()
batch = datos_mmap[:10_000, :]  # accede a 10K filas — SO gestiona el paginado
media_batch = batch.mean(axis=0)
_, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"\n  RAM usada para operar sobre 10K filas: {peak/1024**2:.2f} MB")
print(f"  (el dataset completo serían {datos_mmap.nbytes/1024**3:.2f} GB en RAM)")

# Limpieza
del datos_mmap
os.unlink(temp_path)
print("\n✅ Memory map cerrado y archivo temporal eliminado")

---
## 🧠 Guía de Decisión: Qué técnica usar

```
¿El dataset cabe en RAM?
  ├── SÍ → ¿Es numérico?
  │         ├── SÍ → NumPy con dtype optimizado (float32, int16...)
  │         └── NO → Diccionarios/listas con __slots__ en clases
  └── NO → ¿Necesito acceso aleatorio?
            ├── SÍ → np.memmap (memory mapping)
            └── NO → Generators + procesamiento por batches

¿Millones de instancias de una clase pequeña?
  → Usar __slots__

¿Pipeline de transformaciones en serie?
  → Generators encadenados (O(1) memoria)

¿Modelo entrenado en GPU?
  → float32 obligatorio (float16 para mixed precision)
```

### Reglas de Oro para Producción:
1. **Medir antes de optimizar** (`tracemalloc`, `memory_profiler`)
2. **float32 por defecto** en ML — float64 solo cuando la precisión lo exige
3. **Generators para ETL** — nunca cargar el dataset completo si puedes evitarlo
4. **`del` + `gc.collect()`** después de transformaciones intermedias grandes
5. **`np.memmap`** para datasets >RAM — disponible en Pandas también (`mmap_mode`)